In [ ]:
# Cell 1 — test the Databricks connection
spark.sql("SELECT current_user(), current_catalog()").show()

In [0]:
# Validate that all tables in bronze_dev.redfin have required columns
DEFAULT_KEY = (
    "PERIOD BEGIN",
    "PERIOD END",
    "REGION NAME",
    "LAST UPDATED"
)

# Get all tables in the schema
tables = spark.sql("SHOW TABLES IN bronze_dev.redfin").collect()

print(f"Found {len(tables)} tables in bronze_dev.redfin\n")

# Track results
missing_columns_report = []
valid_tables = []

for table_row in tables:
    table_name = table_row.tableName
    full_table_name = f"bronze_dev.redfin.{table_name}"
    
    # Get the columns for this table
    try:
        columns = spark.sql(f"DESCRIBE TABLE {full_table_name}").collect()
        column_names = [col.col_name.upper() for col in columns if col.col_name and not col.col_name.startswith('#')]
        
        # Check for missing columns
        missing = [col for col in DEFAULT_KEY if col.upper() not in column_names]
        
        if missing:
            missing_columns_report.append({
                "table": full_table_name,
                "missing": missing
            })
            print(f"❌ {full_table_name}")
            print(f"   Missing columns: {', '.join(missing)}\n")
        else:
            valid_tables.append(full_table_name)
            print(f"✅ {full_table_name}")
    except Exception as e:
        print(f"⚠️  Error checking {full_table_name}: {str(e)}\n")

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total tables: {len(tables)}")
print(f"Valid tables (all columns present): {len(valid_tables)}")
print(f"Tables with missing columns: {len(missing_columns_report)}")

if missing_columns_report:
    print("\nTables requiring attention:")
    for item in missing_columns_report:
        print(f"  • {item['table']}: missing {', '.join(item['missing'])}")

In [0]:
from pyspark.sql import Row

# Get all tables in bronze_dev.redfin
tables = spark.sql("SHOW TABLES IN bronze_dev.redfin").collect()

print(f"Processing {len(tables)} tables...\n")

# Collect results
results = []

for table_row in tables:
    table_name = table_row.tableName
    full_table_name = f"bronze_dev.redfin.{table_name}"
    
    try:
        # Get columns for this table
        columns = spark.sql(f"DESCRIBE TABLE {full_table_name}").collect()
        column_names = [col.col_name for col in columns if col.col_name and not col.col_name.startswith('#')]
        
        # For each column, get a sample non-"NA" value
        for col_name in column_names:
            # Query for a non-NA value, limit to 1
            # Cast to string to handle date/numeric columns
            sample_query = f"""
                SELECT `{col_name}` 
                FROM {full_table_name} 
                WHERE `{col_name}` IS NOT NULL 
                  AND CAST(`{col_name}` AS STRING) != 'NA'
                LIMIT 1
            """
            
            sample_result = spark.sql(sample_query).collect()
            
            if sample_result:
                sample_value = sample_result[0][0]
                results.append(Row(
                    table_name=table_name,
                    column_name=col_name,
                    sample_value=str(sample_value) if sample_value is not None else None
                ))
            else:
                # No non-NA values found
                results.append(Row(
                    table_name=table_name,
                    column_name=col_name,
                    sample_value=None
                ))
        
        print(f"✅ Processed {table_name} ({len(column_names)} columns)")
        
    except Exception as e:
        print(f"⚠️  Error processing {full_table_name}: {str(e)}")

# Create DataFrame from results
df = spark.createDataFrame(results)

print(f"\nCreated DataFrame with {df.count()} rows")
display(df)

In [0]:
%sql
SELECT * FROM bronze_dev.redfin.price_drops_county;

In [0]:
%sql
SELECT * FROM silver_dev.INFORMATION_SCHEMA.TABLES WHERE table_schema = 'redfin';

In [0]:
%sql
SELECT distinct column_name 
FROM bronze_dev.INFORMATION_SCHEMA.COLUMNS 
WHERE table_schema = 'redfin' AND table_name NOT LIKE '%materialization%'
ORDER BY 1;

In [0]:
%sql
SELECT * FROM bronze_dev.redfin.delistings_relistings_county
WHERE `TOTAL DELISTINGS` NOT LIKE 'NA' ORDER BY `REGION NAME`;

In [0]:
%sql
SELECT * FROM silver_dev.redfin.delistings_relistings_county
WHERE total_delistings IS NOT NULL
ORDER BY REGION_NAME;